In [1]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
def clean_pass_cat(df):
    table = df.copy()
    table.loc[table['serumPassCat'] == 'EGG', 'serumPassCat'] = '<EGG>'
    table.loc[table['serumPassCat'] == 'CELL', 'serumPassCat'] = '<CELL>'
    table.loc[table['serumPassCat'] == 'BOTH', 'serumPassCat'] = '<BOTH>'

    table.loc[table['virusPassCat'] == 'EGG', 'virusPassCat'] = '<EGG>'
    table.loc[table['virusPassCat'] == 'CELL', 'virusPassCat'] = '<CELL>'
    table.loc[table['virusPassCat'] == 'BOTH', 'virusPassCat'] = '<BOTH>'

    return table
    

In [3]:
Crick_early = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/all.csv')
Crick_41 = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/Crick_41.csv')
Crick_42 = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/Crick_42.csv')
Crick_43 = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/Crick_43.csv')
CDC = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/CDC_all.csv')
CNIC = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/CNIC_all.csv')

Crick_early['institute'] = '<Crick>'
Crick_41['institute'] = '<Crick>'
Crick_42['institute'] = '<Crick>'
Crick_43['institute'] = '<Crick>'
CDC['institute'] = '<CDC>'
CNIC['institute'] = '<CNIC>'

Crick_early['tag'] = 'early'
Crick_41['tag'] = '41'
Crick_42['tag'] = '42'
Crick_43['tag'] = '43'
CDC['tag'] = 'CDC'
CNIC['tag'] = 'CNIC'

All_data = pd.concat([Crick_early, Crick_41, Crick_42, Crick_43, CDC, CNIC])

In [4]:
HA_seqs = pd.concat([All_data['seq_a'], All_data['seq_c']]).unique().tolist()
NA_seqs = pd.concat([All_data['seq_b'], All_data['seq_d']]).unique().tolist()

HA_dict = {key: 'HA_' + str(value) for key, value in zip(HA_seqs, range(len(HA_seqs)))}
NA_dict = {key: 'NA_' + str(value) for key, value in zip(NA_seqs, range(len(NA_seqs)))}

All_data['seq_id_a'] = All_data['seq_a'].map(HA_dict)
All_data['seq_id_b'] = All_data['seq_b'].map(NA_dict)
All_data['seq_id_c'] = All_data['seq_c'].map(HA_dict)
All_data['seq_id_d'] = All_data['seq_d'].map(NA_dict)

All_data = clean_pass_cat(All_data)

In [5]:
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'institute']
agg_dict = {col: 'first' for col in All_data.columns if col not in group_columns}
agg_dict['label'] = 'mean'
All_data = All_data.groupby(group_columns).agg(agg_dict).reset_index()

All_data = All_data[['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d', 'seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'institute', 'tag', 'serumName', 'virusName', 'serumDate', 'virusDate', 'Type','serumIslID', 'virusIslID', 'label', 'serumHA', 'virusHA','seq_diff_mat', 'seq_diff_ohe']]

In [6]:
train_data, test_data = train_test_split(All_data, test_size=0.1, random_state=42)
train_data, valid_data = train_test_split(train_data, test_size=1/9, random_state=42)

In [7]:
serum = train_data[['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b', 'serumPassCat']]
virus = train_data[['seq_id_c', 'seq_id_d', 'seq_c', 'seq_d', 'virusPassCat']]
serum.columns = ['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b', 'virusPassCat']
virus.columns = ['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b', 'virusPassCat']
strain = pd.concat([serum, virus]).drop_duplicates(['seq_a', 'seq_b', 'virusPassCat'])

Artificial_data = pd.concat([strain, strain], axis=1)
Artificial_data.columns = ['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b', 'serumPassCat', 'seq_id_c', 'seq_id_d', 'seq_c', 'seq_d', 'virusPassCat']
Artificial_data = Artificial_data[['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d', 'seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']]

In [8]:
num_values = len(Artificial_data)
proportions = train_data['institute'].value_counts() / len(train_data)
num_value1 = int(num_values * proportions.iloc[0])
num_value2 = int(num_values * proportions.iloc[1])
num_value3 = num_values - num_value1 - num_value2
Institutes = [proportions.index[0]] * num_value1 + [proportions.index[1]] * num_value2 + [proportions.index[2]] * num_value3
random.seed(42)
random.shuffle(Institutes)
Artificial_data['institute'] = Institutes
Artificial_data['label'] = 0

In [11]:
All_data.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/fluProfiler_ultra/All_data.csv', index=False)
train_data.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/fluProfiler_ultra/train.csv', index=False)
valid_data.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/fluProfiler_ultra/valid.csv', index=False)
test_data.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/fluProfiler_ultra/test.csv', index=False)
Artificial_data.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/fluProfiler_ultra/Artificial_data.csv', index=False)